In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

dbutils.widgets.text("batch_id", "batch_002")

batch_id = dbutils.widgets.get("batch_id")

print("Quality checking batch:", batch_id)

In [0]:
customers = (
    spark.table("bank_catalog.silver.customers")
    .filter(F.col("batch_id") == batch_id)
)

accounts = (
    spark.table("bank_catalog.silver.accounts")
    .filter(F.col("batch_id") == batch_id)
)

transactions = (
    spark.table("bank_catalog.silver.transactions")
    .filter(F.col("batch_id") == batch_id)
)

loans = (
    spark.table("bank_catalog.silver.loans")
    .filter(F.col("batch_id") == batch_id)
)

branches = (
    spark.table("bank_catalog.silver.branches")
    .filter(F.col("batch_id") == batch_id)
)

In [0]:
all_customers = spark.table(
    "bank_catalog.silver.customers"
)

all_accounts = spark.table(
    "bank_catalog.silver.accounts"
)

In [0]:
quality_results = []

def run_check(df, table_name, check_name, condition):

    total_records = df.count()

    failed_records = df.filter(condition).count()

    passed_records = total_records - failed_records

    status = (
        "PASS"
        if failed_records == 0
        else "FAIL"
    )

    quality_results.append({
        "batch_id": batch_id,
        "table_name": table_name,
        "check_name": check_name,
        "total_records": total_records,
        "passed_records": passed_records,
        "failed_records": failed_records,
        "status": status
    })

    print(
        f"{table_name} | {check_name} | "
        f"{status} | Failed: {failed_records}"
    )

In [0]:
run_check(
    customers,
    "customers",
    "Customer ID NULL",
    F.col("customer_id").isNull()
)

run_check(
    customers,
    "customers",
    "Customer DOB NULL",
    F.col("date_of_birth").isNull()
)

run_check(
    customers,
    "customers",
    "Customer name NULL",
    F.col("first_name").isNull() |
    F.col("last_name").isNull()
)

In [0]:
run_check(
    accounts,
    "accounts",
    "Account ID NULL",
    F.col("account_id").isNull()
)

run_check(
    accounts,
    "accounts",
    "Customer ID NULL",
    F.col("customer_id").isNull()
)

run_check(
    accounts,
    "accounts",
    "Account balance NULL",
    F.col("current_balance").isNull()
)

run_check(
    accounts,
    "accounts",
    "Invalid account status",
    ~F.col("account_status").isin(
        "ACTIVE", "CLOSED", "FROZEN"
    ) | F.col("account_status").isNull()
)

In [0]:
run_check(
    transactions,
    "transactions",
    "Transaction ID NULL",
    F.col("transaction_id").isNull()
)

run_check(
    transactions,
    "transactions",
    "Negative transaction amount",
    F.col("amount") < 0
)

run_check(
    transactions,
    "transactions",
    "Transaction amount NULL",
    F.col("amount").isNull()
)

run_check(
    transactions,
    "transactions",
    "Invalid transaction date",
    F.col("transaction_timestamp").isNull()
)

run_check(
    transactions,
    "transactions",
    "Invalid transaction status",
    ~F.col("transaction_status").isin(
        "SUCCESS", "FAILED", "PENDING"
    ) | F.col("transaction_status").isNull()
)

In [0]:
run_check(
    loans,
    "loans",
    "Loan ID NULL",
    F.col("loan_id").isNull()
)

run_check(
    loans,
    "loans",
    "Invalid sanctioned amount",
    F.col("sanctioned_amount").isNull() |
    (F.col("sanctioned_amount") <= 0)
)

run_check(
    loans,
    "loans",
    "Invalid interest rate",
    F.col("interest_rate").isNull() |
    ~F.col("interest_rate").between(0, 30)
)

run_check(
    loans,
    "loans",
    "Invalid tenure",
    F.col("tenure_months").isNull() |
    (F.col("tenure_months") <= 0)
)

In [0]:
invalid_accounts = accounts.join(
    all_customers.select("customer_id").distinct(),
    "customer_id",
    "left_anti"
)

run_check(
    invalid_accounts,
    "accounts",
    "Invalid customer reference",
    F.lit(True)
)

In [0]:
invalid_transactions = transactions.join(
    all_accounts.select("account_id").distinct(),
    "account_id",
    "left_anti"
)

run_check(
    invalid_transactions,
    "transactions",
    "Invalid account reference",
    F.lit(True)
)

In [0]:
invalid_loans = loans.join(
    all_customers.select("customer_id").distinct(),
    "customer_id",
    "left_anti"
)

run_check(
    invalid_loans,
    "loans",
    "Invalid customer reference",
    F.lit(True)
)

In [0]:
quality_schema = """
batch_id STRING,
table_name STRING,
check_name STRING,
total_records LONG,
passed_records LONG,
failed_records LONG,
status STRING
"""

quality_df = spark.createDataFrame(
    quality_results,
    schema=quality_schema
)

quality_df = quality_df.withColumn(
    "audit_timestamp",
    F.current_timestamp()
)

quality_df = quality_df.withColumn(
    "run_id",
    F.lit(batch_id)
)

display(quality_df)

In [0]:
existing = {
    r["col_name"] for r in spark.sql(
        "DESCRIBE TABLE bank_catalog.quarantine.data_quality_audit"
    ).collect()
    if not r["col_name"].startswith("#")
}

missing = [c for c in ("batch_id", "passed_records") if c not in existing]

if missing:
    types = {"batch_id": "STRING", "passed_records": "BIGINT"}
    spark.sql(
        "ALTER TABLE bank_catalog.quarantine.data_quality_audit "
        "ADD COLUMNS (" + ", ".join(f"{c} {types[c]}" for c in missing) + ")"
    )

In [0]:
from delta.tables import DeltaTable

quality_target = "bank_catalog.quarantine.data_quality_audit"

if not spark.catalog.tableExists(quality_target):

    quality_df.write \
        .format("delta") \
        .mode("overwrite") \
        .saveAsTable(quality_target)

else:

    delta_table = DeltaTable.forName(
        spark, quality_target
    )

    (
        delta_table.alias("target")
        .merge(
            quality_df.alias("source"),
            """
            target.batch_id = source.batch_id
            AND target.table_name = source.table_name
            AND target.check_name = source.check_name
            """
        )
        .whenMatchedUpdate(set={
            "total_records": "source.total_records",
            "passed_records": "source.passed_records",
            "failed_records": "source.failed_records",
            "status": "source.status",
            "audit_timestamp": "source.audit_timestamp",
            "run_id": "source.run_id"
        })
        .whenNotMatchedInsertAll()
        .execute()
    )

print("Incremental quality audit completed!")

In [0]:
def run_reference_check(
    source_df,
    invalid_df,
    table_name,
    check_name
):

    total_records = source_df.count()
    failed_records = invalid_df.count()

    quality_results.append({
        "batch_id": batch_id,
        "table_name": table_name,
        "check_name": check_name,
        "total_records": total_records,
        "passed_records": total_records - failed_records,
        "failed_records": failed_records,
        "status": "PASS" if failed_records == 0 else "FAIL"
    })

In [0]:
run_reference_check(
    accounts,
    invalid_accounts,
    "accounts",
    "Invalid customer reference"
)

run_reference_check(
    transactions,
    invalid_transactions,
    "transactions",
    "Invalid account reference"
)

run_reference_check(
    loans,
    invalid_loans,
    "loans",
    "Invalid customer reference"
)

In [0]:
%sql
SELECT
    batch_id,
    table_name,
    COUNT(*) AS total_checks,
    SUM(failed_records) AS total_failures
FROM bank_catalog.quarantine.data_quality_audit
GROUP BY batch_id, table_name
ORDER BY batch_id, table_name;

In [0]:
%sql
SELECT
    batch_id,
    table_name,
    check_name,
    COUNT(*) AS duplicate_count
FROM bank_catalog.quarantine.data_quality_audit
GROUP BY batch_id, table_name, check_name
HAVING COUNT(*) > 1;